# Pendientes del proyecto — Clasificación automática de tipos celulares en scRNA-seq

Este notebook complementa `exploracion.ipynb` con los elementos faltantes identificados en la evaluación de la rúbrica CRISP-DM.

**Pendientes cubiertos:**
1. Referencia del dataset
2. Visualizaciones faltantes (UMAP por lineage + distribución de clases)
3. Preprocesamiento implementado en código
4. Data Quality Report consolidado
5. Diagrama de arquitectura del pipeline

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print('matplotlib:', matplotlib.__version__)

file_path = Path('..') / 'data' / 'adata_final.h5ad'
adata = ad.read_h5ad(file_path)
print('Dataset cargado:', adata.shape)

---
## 1. Referencia del dataset

| Campo | Detalle |
|---|---|
| Nombre del archivo | `adata_final.h5ad` |
| Origen | Contacto académico de la Universidad Tecnológica Metropolitana (UTEM) |
| Tipo de fuente | Dataset proporcionado directamente por investigador/contacto institucional |
| Publicación / DOI | No disponible públicamente |
| Acceso | Restringido — proporcionado para uso académico en el contexto de este análisis |

**Descripción técnica del dataset:**
- Formato: AnnData (`.h5ad`)
- Células: 328,170
- Genes (highly variable): 2,000
- Variable objetivo: `lineage` (6 clases: T, MNP, B&plasma, NK, mast, pDC)
- Preprocesamiento previo documentado en el objeto AnnData: normalización log1p (`adata.uns['log1p']`), selección de HVG, PCA (50 componentes), UMAP, grafo de vecinos KNN
- Dominio: Inmunología / transcriptómica de célula única (scRNA-seq)
- Muestras de origen: 91 muestras distintas (`sample_ID`)

> **Nota:** Al ser un dataset de origen académico no publicado, no existe DOI ni URL de descarga pública. Su uso está enmarcado en el contexto del presente análisis con fines educativos e investigativos.

---
## 2. Visualizaciones faltantes

### 2.1 Distribución de clases (`lineage`)

In [ ]:
lineage_counts = adata.obs['lineage'].value_counts().sort_values(ascending=False)
lineage_pct = (lineage_counts / lineage_counts.sum() * 100).round(2)
n_classes = len(lineage_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot de conteos
axes[0].bar(lineage_counts.index, lineage_counts.values,
            color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de clases — lineage (conteo)', fontsize=13)
axes[0].set_xlabel('Tipo celular')
axes[0].set_ylabel('Número de células')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(lineage_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

# Pie chart — colores generados dinámicamente
pie_colors = [plt.cm.Set2(i / n_classes) for i in range(n_classes)]
axes[1].pie(lineage_pct.values, labels=lineage_pct.index,
            autopct='%1.1f%%', startangle=140, colors=pie_colors)
axes[1].set_title('Distribución de clases — lineage (%)', fontsize=13)

plt.tight_layout()
plt.show()

print('\nConteo y porcentaje por clase:')
df_dist = pd.DataFrame({'Conteo': lineage_counts, 'Porcentaje (%)': lineage_pct})
print(df_dist.to_string())

**Interpretación:**  
El gráfico evidencia el desbalance de clases en el dataset. Las clases mayoritarias (T, MNP) concentran la mayor parte de las células, mientras que pDC es la clase más minoritaria. Este desbalance justifica el uso de métricas como F1-score macro y recall por clase, y la necesidad de estrategias de balanceo durante el modelado.

### 2.2 UMAP coloreado por `lineage`

In [ ]:
umap_coords = adata.obsm['X_umap']
lineage_labels = adata.obs['lineage'].astype(str).values
unique_classes = sorted(set(lineage_labels))
n_cls = len(unique_classes)

# Acceso directo al colormap — compatible con matplotlib >= 3.7
cmap = matplotlib.colormaps['tab10']
color_map = {cls: cmap(i / n_cls) for i, cls in enumerate(unique_classes)}
colors = [color_map[l] for l in lineage_labels]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
           c=colors, s=0.3, alpha=0.4, rasterized=True)

legend_handles = [
    mpatches.Patch(color=color_map[cls], label=cls)
    for cls in unique_classes
]
ax.legend(handles=legend_handles, title='Lineage',
          bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_title('UMAP coloreado por tipo celular (lineage)', fontsize=14)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()

**Interpretación:**  
El UMAP muestra la separación espacial de los tipos celulares en el espacio de baja dimensión. Clusters bien definidos indican que los perfiles de expresión génica son suficientemente distintos entre tipos celulares, lo que valida la viabilidad del problema de clasificación. Clusters solapados señalan tipos celulares con perfiles similares, que serán más difíciles de clasificar.

---
## 3. Preprocesamiento implementado en código

Se implementa el pipeline de preprocesamiento completo, dejando los datos listos para modelado.

In [ ]:
# --- 3.1 Extracción de features y variable objetivo ---
X = adata.obsm['X_pca']          # shape (328170, 50)
y_raw = adata.obs['lineage'].astype(str).values

print('Shape de features (X_pca):', X.shape)
print('Shape de target (lineage):', y_raw.shape)
print('Clases únicas:', np.unique(y_raw))

In [ ]:
# --- 3.2 Encoding de la variable objetivo ---
le = LabelEncoder()
y = le.fit_transform(y_raw)

print('Clases codificadas:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')

In [ ]:
# --- 3.3 Train/Test split estratificado ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Tamaño del conjunto de entrenamiento:', X_train.shape)
print('Tamaño del conjunto de prueba:       ', X_test.shape)

label_map = dict(enumerate(le.classes_))

print('\nDistribución de clases en train:')
train_dist = pd.Series(y_train).map(label_map).value_counts(normalize=True).round(4)
print(train_dist.to_string())

print('\nDistribución de clases en test:')
test_dist = pd.Series(y_test).map(label_map).value_counts(normalize=True).round(4)
print(test_dist.to_string())

**Justificación técnica de las decisiones de preprocesamiento:**

| Decisión | Justificación |
|---|---|
| Usar `X_pca` (50 componentes) como features | La matriz original (328K × 2000) es dispersa (~8.2% densidad). PCA reduce dimensionalidad preservando ~49% de varianza, elimina ruido y hace el modelado computacionalmente viable |
| No normalizar `X_pca` adicional | PCA ya fue calculado sobre datos normalizados (log1p). Las coordenadas PCA están en escala comparable |
| LabelEncoder para `lineage` | Variable categórica nominal con 6 clases. LabelEncoder es adecuado para clasificadores que aceptan enteros como target (sklearn) |
| Split estratificado 80/20 | Mantiene la proporción de clases en train y test, crítico dado el desbalance. 20% de test sobre 328K células = ~65K muestras, suficiente para evaluación robusta |
| random_state=42 | Garantiza reproducibilidad del experimento |

---
## 4. Data Quality Report consolidado

In [ ]:
nnz      = adata.X.nnz
total    = adata.n_obs * adata.n_vars
densidad = nnz / total
sparsity = 1 - densidad
nulls_obs  = adata.obs.isnull().sum().sum()
nulls_var  = adata.var.isnull().sum().sum()
n_dup_obs  = adata.obs.duplicated().sum()

sep = '=' * 55
print(sep)
print('         DATA QUALITY REPORT')
print(sep)
print(f'  Células (observaciones) : {adata.n_obs:>10,}')
print(f'  Genes (variables)       : {adata.n_vars:>10,}')
print(f'  Shape                   : {str(adata.shape):>10}')
print('-' * 55)
print(f'  Valores nulos en obs    : {nulls_obs:>10}')
print(f'  Valores nulos en var    : {nulls_var:>10}')
print(f'  Completitud             : {"100%":>10}')
print('-' * 55)
print(f'  Valores no nulos (nnz)  : {nnz:>10,}')
print(f'  Densidad de la matriz X : {densidad:>10.4f}  (~8.2%)')
print(f'  Sparsity                : {sparsity:>10.4f}  (~91.8%)')
print('-' * 55)
print(f'  Filas duplicadas (obs)  : {n_dup_obs:>10}')
print('-' * 55)
print('  Distribución de clases (lineage):')
for cls, cnt in lineage_counts.items():
    pct = cnt / lineage_counts.sum() * 100
    print(f'    {cls:<20} {cnt:>8,}  ({pct:.1f}%)')
print('-' * 55)
print(f'  Tipo de matriz X        : {type(adata.X).__name__:>10}')
print(f'  Embeddings disponibles  : {str(list(adata.obsm.keys()))}')
print(sep)

In [ ]:
riesgos = [
    ('Desbalance de clases',
     'Clases minoritarias (pDC, mast) vs. mayoritarias (T, MNP). '
     'Riesgo: el modelo puede ignorar clases minoritarias. '
     "Mitigación: usar class_weight='balanced', SMOTE, o métricas macro."),
    ('Alta sparsity (~91.8%)',
     'La mayoría de valores en la matriz de expresión son cero (dropouts). '
     'Riesgo: modelos sensibles a escala pueden verse afectados. '
     'Mitigación: usar PCA (ya aplicado).'),
    ('Fuente del dataset no publicada',
     'El dataset fue proporcionado por un contacto académico de la UTEM. '
     'No tiene DOI ni URL pública. '
     'Riesgo: no se puede reproducir la generación del archivo externamente. '
     'Mitigación: el archivo .h5ad contiene el preprocesamiento documentado internamente.'),
    ('PCA captura solo ~49% de varianza',
     'Con 50 componentes se retiene ~mitad de la varianza original. '
     'Riesgo: pérdida de información biológica. '
     'Mitigación: evaluar si aumentar componentes mejora el modelo.'),
    ('Sesgo de batch',
     'El dataset proviene de 91 muestras (sample_ID). '
     'Riesgo: efectos de batch pueden introducir variabilidad técnica. '
     'Mitigación: HVG seleccionados por batch (highly_variable_nbatches).'),
]

print('RIESGOS Y LIMITACIONES DEL DATASET')
print('=' * 60)
for i, (riesgo, descripcion) in enumerate(riesgos, 1):
    print(f'\n{i}. {riesgo}')
    print(f'   {descripcion}')
print('\n' + '=' * 60)

---
## 5. Diagrama de arquitectura del pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

stage_colors = {
    'fuente':  '#4A90D9',
    'raw':     '#7B68EE',
    'curated': '#2ECC71',
    'proc':    '#F39C12',
    'modelo':  '#E74C3C',
    'consumo': '#1ABC9C',
}

def draw_box(ax, x, y, w, h, color, title, lines, fontsize=8.5):
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h, boxstyle='round,pad=0.1',
        linewidth=1.5, edgecolor='white', facecolor=color, alpha=0.88
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.22, title,
            ha='center', va='top', fontsize=10, fontweight='bold', color='white')
    for idx, line in enumerate(lines):
        ax.text(x + w/2, y + h - 0.55 - idx*0.32, line,
                ha='center', va='top', fontsize=fontsize, color='white')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->',
                                color='#555555', lw=2.0,
                                connectionstyle='arc3,rad=0.0'))

draw_box(ax, 0.3, 6.8, 3.0, 1.8, stage_colors['fuente'], '1. FUENTE',
         ['Estudio scRNA-seq', 'Atlas inmunologico', 'Fuente: UTEM (academico)', 'Formato: .h5ad (AnnData)'])
draw_box(ax, 0.3, 4.5, 3.0, 1.9, stage_colors['raw'], '2. RAW',
         ['328,170 celulas x 2,000 genes', 'Matriz dispersa (CSC)', 'Densidad: ~8.2%', 'Metadatos: obs + var'])
draw_box(ax, 0.3, 2.2, 3.0, 1.9, stage_colors['curated'], '3. CURATED',
         ['Normalizacion log1p', 'Seleccion HVG (2,000 genes)', 'PCA (50 componentes)', 'UMAP + KNN graph'])
draw_box(ax, 4.5, 4.5, 4.5, 3.2, stage_colors['proc'], '4. PREPROCESAMIENTO',
         ['Features: X_pca (328K x 50)', 'Target: lineage (6 clases)',
          'Encoding: LabelEncoder', 'Split: 80% train / 20% test',
          'Estratificado por clase', 'random_state=42'])
draw_box(ax, 4.5, 1.8, 4.5, 2.3, stage_colors['modelo'], '5. MODELADO',
         ['Clasificador supervisado', 'Manejo de desbalance (class_weight)',
          'Metricas: Accuracy, F1-macro,', 'Recall por clase, Conf. Matrix'])
draw_box(ax, 10.0, 3.5, 3.5, 2.5, stage_colors['consumo'], '6. CONSUMO',
         ['Prediccion de tipo celular', 'Analisis de composicion',
          'Soporte a investigacion', 'clinica / biologica'])

draw_arrow(ax, 1.8, 6.8, 1.8, 6.4)
draw_arrow(ax, 1.8, 4.5, 1.8, 4.1)
draw_arrow(ax, 3.3, 5.5, 4.5, 5.5)
draw_arrow(ax, 6.75, 4.5, 6.75, 4.1)
draw_arrow(ax, 9.0, 3.5, 10.0, 4.5)

ax.text(7, 8.7,
        'Arquitectura del Pipeline — Clasificacion de Tipos Celulares (scRNA-seq)',
        ha='center', va='top', fontsize=12, fontweight='bold', color='#2c3e50')
ax.text(0.3, 1.5,
        'Reproducibilidad: random_state=42 | Datos: adata_final.h5ad | Version: 1.0',
        fontsize=7.5, color='#555555', style='italic')

plt.tight_layout()
plt.show()

**Descripción del pipeline:**

| Etapa | Descripción | Artefacto |
|---|---|---|
| 1. Fuente | Dataset original de estudio scRNA-seq | `.h5ad` (AnnData) |
| 2. Raw | Datos crudos: matriz de expresión dispersa + metadatos | `adata.X`, `adata.obs`, `adata.var` |
| 3. Curated | Preprocesamiento biológico previo: log1p, HVG, PCA, UMAP | `adata.obsm['X_pca']`, `adata.obsm['X_umap']` |
| 4. Preprocesamiento | Extracción de features, encoding, split estratificado | `X_train`, `X_test`, `y_train`, `y_test` |
| 5. Modelado | Entrenamiento y evaluación del clasificador | Modelo entrenado + métricas |
| 6. Consumo | Predicción de tipos celulares en nuevas muestras | Clasificación automática |

**Criterios de reproducibilidad:**
- `random_state=42` en todos los pasos estocásticos
- Datos fijos: `adata_final.h5ad` (no se modifica el archivo original)
- Split estratificado garantiza consistencia entre ejecuciones